# Qwen3-Coder-Next UD-Q3_K_XL Inference

Simple Colab A100 notebook. Uses the Unsloth GGUF quant `UD-Q3_K_XL` (~36.3 GB), loads 1k test rows from Hugging Face, and prints macro F1 when labels exist.

In [1]:
DATASET = "DaniilOr/SemEval-2026-Task13"
CONFIG = "A"
MODEL_REPO = "unsloth/Qwen3-Coder-Next-GGUF"
MODEL_FILE = "Qwen3-Coder-Next-UD-Q3_K_XL.gguf"
MODEL_PATH = f"/content/model/{MODEL_FILE}"

N_ROWS = 1000
N_SHOTS_PER_LABEL = 4
MAX_CODE_CHARS = 2200
MAX_SHOT_CHARS = 700
CTX_SIZE = 2048
PORT = 8001

## Install and Download

In [2]:
!nvidia-smi
!rm -rf /root/.cache/huggingface /content/sample_data
!apt-get update -y >/dev/null
!apt-get install -y build-essential cmake curl libcurl4-openssl-dev git >/dev/null
import sys
!{sys.executable} -m pip uninstall -y -q datasets
!{sys.executable} -m pip install -q --no-cache-dir "datasets==4.3.0" "huggingface_hub>=0.34.0" openai pandas pyarrow tqdm scikit-learn requests
!{sys.executable} -c "import datasets; print('datasets', datasets.__version__)"

from pathlib import Path
from huggingface_hub import hf_hub_download

Path("/content/model").mkdir(exist_ok=True)
hf_hub_download(MODEL_REPO, filename=MODEL_FILE, local_dir="/content/model")

if not Path("/content/llama.cpp").exists():
    !git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DBUILD_SHARED_LIBS=OFF >/dev/null
!cmake --build /content/llama.cpp/build --config Release -j --target llama-server >/dev/null

!du -sh /content/model /content/llama.cpp || true
!df -h /content

Fri May 29 17:59:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             47W /  400W |   35522MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


CMAKE_BUILD_TYPE=Release
34G	/content/model
1.3G	/content/llama.cpp
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   83G   31G  74% /


## Start Model

In [3]:
import socket, subprocess, time, requests


def up():
    s = socket.socket(); s.settimeout(1)
    ok = s.connect_ex(("127.0.0.1", PORT)) == 0
    s.close()
    return ok


if not up():
    cmd = [
        "/content/llama.cpp/build/bin/llama-server",
        "--model", MODEL_PATH,
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--ctx-size", str(CTX_SIZE),
        "--n-gpu-layers", "-1",
        "--temp", "0",
        "--top-k", "1",
    ]
    server = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    for _ in range(180):
        if up(): break
        time.sleep(1)
    assert up(), "Server port did not open. Try a smaller quant."

for _ in range(300):
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=2)
        if r.status_code == 200:
            print("model ready")
            break
        print(r.status_code, r.text[:120])
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Model API not ready after 10 minutes. Try a smaller quant.")

print(f"ready on http://127.0.0.1:{PORT}/v1")


model ready
ready on http://127.0.0.1:8001/v1


## Load Data

In [4]:
import pandas as pd
from datasets import load_dataset

# Expected columns: code:string, generator:string, label:int64, language:string.
# Use only code + language for prompting. Use label only for scoring.
test = load_dataset(DATASET, CONFIG, split="test").to_pandas().head(N_ROWS).copy()
required = {"code", "label", "language"}
missing = required - set(test.columns)
assert not missing, f"Missing required columns: {missing}"

try:
    shot_source = load_dataset(DATASET, CONFIG, split="trial").to_pandas()
except Exception:
    shot_source = load_dataset(DATASET, CONFIG, split="train[:2000]").to_pandas()

shot_source = shot_source.copy()
shot_source["code_len"] = shot_source["code"].astype(str).str.len()

shots = []
for label in [0, 1]:
    pool = shot_source[shot_source.label.astype(int) == label]
    compact = pool[(pool.code_len >= 80) & (pool.code_len <= 1400)]
    if len(compact) < N_SHOTS_PER_LABEL:
        compact = pool
    shots += compact.sample(min(N_SHOTS_PER_LABEL, len(compact)), random_state=3407).to_dict("records")

print("test shape:", test.shape)
print("columns:", test.columns.tolist())
print("label distribution:", test["label"].astype(int).value_counts().sort_index().to_dict())
print("few-shot labels:", {0: sum(int(s["label"]) == 0 for s in shots), 1: sum(int(s["label"]) == 1 for s in shots)})
test.head(2)


README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

task_a/task_a_validation_set.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

task_a/task_a_test_set_sample.parquet:   0%|          | 0.00/593k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

test shape: (1000, 4)
columns: ['code', 'generator', 'label', 'language']
label distribution: {0: 777, 1: 223}
few-shot labels: {0: 4, 1: 4}


,code,generator,label,language
0,public Vector To(Vector o)\n {\n ...,Human,0,C#
1,func (v *DefaultMessageSyntaxValidator) Valida...,Human,0,Go


## Inference and F1

In [5]:
import json, re
from openai import OpenAI
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, f1_score

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="not-needed")

SYSTEM = """You are a careful detector for SemEval Task A: human-written vs machine-generated code.

Return only valid JSON: {"label": 0 or 1, "confidence": 0.0 to 1.0}.
0 = human-written. 1 = machine-generated.

Decision policy:
- Predict 1 only when there is clear evidence of machine generation.
- Normal working code, terse contest code, project-specific tests, odd formatting, bugs, or low readability are not enough to call AI.
- Clean style, comments, React components, or boilerplate are not enough to call AI.
- Strong AI signals include explanatory prose inside code, markdown/code-fence artifacts, generic over-explained comments, placeholder examples, inconsistent APIs/imports, unused parameters caused by templating, impossible variables, and code that describes a solution more than implementing it.
- If evidence is weak or ambiguous, choose label 0.
"""

def snippet(row, max_chars=MAX_CODE_CHARS):
    code = str(row.get("code", ""))[:max_chars]
    return f"Language: {row.get('language', 'unknown')}\nCode:\n```\n{code}\n```"

def messages(row):
    out = [{"role": "system", "content": SYSTEM}]
    for s in shots:
        out += [
            {"role": "user", "content": snippet(s, MAX_SHOT_CHARS)},
            {"role": "assistant", "content": json.dumps({"label": int(s["label"]), "confidence": 0.9})},
        ]
    out.append({"role": "user", "content": snippet(row) + "\n\nClassify this snippet. Return JSON only."})
    return out

def parse(text):
    m = re.search(r"\{.*?\}", text, re.S)
    if not m:
        return 0, 0.0, text
    try:
        obj = json.loads(m.group(0))
        label = int(obj.get("label", 0))
        confidence = max(0.0, min(1.0, float(obj.get("confidence", 0.0))))
        return int(label == 1), confidence, text
    except Exception:
        return 0, 0.0, text

preds, confs, raws = [], [], []
for row in tqdm(test.to_dict("records")):
    r = client.chat.completions.create(model="local", messages=messages(row), temperature=0, max_tokens=32)
    label, conf, raw = parse(r.choices[0].message.content or "")
    preds.append(label); confs.append(conf); raws.append(raw)

test["prediction"] = preds
test["confidence"] = confs
test["raw_model_output"] = raws
test.to_csv("/content/qwen3_coder_next_ud_q3_k_xl_predictions.csv", index=False)

id_col = "ID" if "ID" in test.columns else "id" if "id" in test.columns else None
submission = pd.DataFrame({"ID": test[id_col] if id_col else range(len(test)), "label": preds})
submission.to_csv("/content/qwen3_coder_next_ud_q3_k_xl_submission.csv", index=False)

y_true = test["label"].astype(int).tolist()
y_pred = [int(x) for x in preds]
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, digits=4, target_names=["human", "machine"]))


  0%|          | 0/1000 [00:00<?, ?it/s]

Macro F1: 0.55907917165955
              precision    recall  f1-score   support

       human     0.8980    0.5212    0.6596       777
     machine     0.3224    0.7937    0.4585       223

    accuracy                         0.5820      1000
   macro avg     0.6102    0.6575    0.5591      1000
weighted avg     0.7696    0.5820    0.6148      1000

